# Single Cell Analysis Pipeline Workflow

This notebook demonstrates the end-to-end workflow for high-content imaging analysis using the `Analysis_Pipeline_Tool`.<br>
This notebook is designed to be used after the Preanalysis Workflow notebook

## Pipeline Overview
1. **Experiment Planning**: Create plate layout.
2. **Preprocessing**: Filter for focus and generate previews.
3. **Segmentation**: Identify cells (CellPose).
4. **Aggregation**: Merge well-level data.
5. **Metadata**: Annotate data with experimental conditions.
6. **Cleaning & Normalization**: Prepare for stats.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys
import pandas as pd
from pathlib import Path
import gc

# Ensure src is in path
sys.path.append('src')

from src.config import Config
from src.planning import PlateMapGenerator
from src.preprocessing import FocusAnalyzer, PreviewGenerator
from src.visualization import PlateHeatmap, SegmentationGenerator
from src.aggregation import DataAggregator
from src.metadata import MetadataMerger
from src.cleaning import DataCleaner
from src.normalization import run_normalization

print("Pipeline Modules Loaded Successfully.")

Pipeline Modules Loaded Successfully.


# Data loading

In [ ]:
from src.data_loader_ui import DataLoaderUI
loader = DataLoaderUI()
loader.display()

## Makes usable dataframes for downstream analysis
### Need to run below cell to save DFs to environment

In [ ]:
df_cell, df_image = loader.get_data()

### I reccommend saving dfs as pickles or parquets after for easier loading in case kernel crashes

In [ ]:
df_cell.to_pickle('C:\Matthew_Analysis\Airway_Analysis_Classification\TSZ_Screen\df_cell.pkl')

In [ ]:
df_cell = pd.read_pickle(r'C:\Matthew_Analysis\Airway_Analysis_Classification\TSZ_Screen\df_cell.pkl')

# Metadata Merging and CQ1 naming conversion

## You must click the df_image = widgets.merged before starting the secondary merge

## Instructions
After loading in data form either a csv or sqlite database attach metadata to image level data. <br> Provide the column names for the merge keys. If it was imaged on the CQ1 can toggel conversion of CQ1 nameing to standard (A1, A2, etc.) naming scheme. <br>
After merging metadata onto image level data can merge it onto the cell level data as the secondary df. <br>
You will most likely use image number for this.

In [ ]:
from src.metadata_merging_ui import MetadataMergeWidget
# Assuming you have a dataframe 'my_df'
widget = MetadataMergeWidget(input_df=df_image, secondary_df=df_cell)

widget.display()

## Instruction
You need to save each DF to memory individually after mergin. Specify the name below. Do this once after merging the primary DF then again after merging onto the secondary DF

In [ ]:
# User interacts
df_image = widget.merged_df

In [ ]:
df_cell = widget.merged_df

## If you saved metadata to cell information you will get duplicate columns with _x and _y at the end of them. This cell will fix that

In [ ]:
import pandas as pd
import numpy as np

def remove_duplicate_merged_columns(df_input):
    """
    Scans for _x and _y pairs. If data matches, drops _y and renames _x.
    Handles NaN values and non-unique column names safely.
    """
    # 1. Create a copy to avoid 'SettingWithCopy' warnings
    temp_df = df_input.copy()
    
    # 2. Fix non-unique column names if they exist (prevents the ValueError you saw)
    if not temp_df.columns.is_unique:
        new_cols = []
        counts = {}
        for col in temp_df.columns:
            if col in counts:
                counts[col] += 1
                new_cols.append(f"{col}.{counts[col]}")
            else:
                counts[col] = 0
                new_cols.append(col)
        temp_df.columns = new_cols

    # 3. Identify pairs ending in _x
    x_cols = [c for c in temp_df.columns if str(c).endswith('_x')]
    cols_to_drop = []
    rename_map = {}

    for col_x in x_cols:
        base_name = col_x[:-2]
        col_y = base_name + '_y'
        
        # If the matching _y exists, compare them
        if col_y in temp_df.columns:
            # .equals() is vital: it treats NaN == NaN as True
            if temp_df[col_x].equals(temp_df[col_y]):
                cols_to_drop.append(col_y)
                rename_map[col_x] = base_name
            else:
                print(f"⚠️ Mismatch in '{base_name}': Keeping both _x and _y versions.")

    # 4. Apply changes
    temp_df = temp_df.drop(columns=cols_to_drop)
    temp_df = temp_df.rename(columns=rename_map)
    
    return temp_df

df = df_cell

# ==========================================
# EXECUTION
# ==========================================
# This assigns the cleaned version back to 'df' 
# making it accessible to all other cells in your notebook.
df_cleaned = remove_duplicate_merged_columns(df)

print("Cleanup complete. Current columns in 'df':")
print(df.columns.tolist())
df.head()
del df

# Normalization
### I havent used this much it is probably buggy

In [ ]:
from src.normalization_widget import NormalizationWidget
# Pass your merged DataFrame directly
norm_ui = NormalizationWidget(input_df=df_cell)
norm_ui.display()

## Saves Normalized data as DF

In [ ]:
normalized_df = norm_ui.normalized_df
normalized_df.head()

## Optional subsetting for faster downstream work

In [ ]:
# Assuming your dataframe is named 'df'
subset_df = normalized_df.sample(frac=0.33, random_state=42)

# Optional: Display the new shape to verify
print(f"Original shape: {normalized_df.shape}")
print(f"Subset shape: {subset_df.shape}")
subset_df.head()

# Data filtering
### This is really only set up for cell level data at this point

In [ ]:
# Cleaning - Interactive Widget
from src.data_filtering_widget import DataFilteringWidget
# Initialize and display the widget
filter_widget = DataFilteringWidget()
print("Use the widget below to filter your data. Recommmended Output Name: 'filtered_df'")
filter_widget.display()

### Again I recommend saving this df as a pickle for easy reloading

In [ ]:
filtered_df.to_pickle('C:\Matthew_Analysis\Airway_Analysis_Classification\TSZ_Screen\df_filtered.pkl')

In [2]:
filtered_df = pd.read_pickle(r"C:\Matthew_Analysis\Airway_Analysis_Classification\TSZ_Screen\df_filtered.pkl")

In [ ]:
necrotopic_only = filtered_df[filtered_df['Metadata_State'] == 'Necrotopic']


# UMAP Generation and Exploration

### The csv_path is not actually loaded and is a relic of old code but keep something there to prevent errors
### You must include a path to your images (I dont reccomment keeping them on the NAS use either Turbo or local storage)
### The script will generate a subfolder called UMAP that it will save everything too. 
### You can specify either the cell or nucleus bounding boxes for the center of the images.

In [2]:
# Create NotebookConfig
from src.tile_extraction import NotebookConfig
config = NotebookConfig(
    # Data Paths
    csv_path=r"C:\Matthew_Analysis\Airway_Analysis_Classification\TSZ_Screen\CP_CSVs\master_single_cell.csv",
    image_base_path=r"D:\Images\MMC_0023\20260217T092025",
    output_dir='./UMAP',
    
    # Bounding Box Columns (Use these instead of x/y_column)
    bbox_min_x_column='Nucleus_AreaShape_BoundingBoxMinimum_X',
    bbox_max_x_column='Nucleus_AreaShape_BoundingBoxMaximum_X',
    bbox_min_y_column='Nucleus_AreaShape_BoundingBoxMinimum_Y',
    bbox_max_y_column='Nucleus_AreaShape_BoundingBoxMaximum_Y',
    
    # Optional: Center columns (can be None if using bbox)
    x_column=None,
    y_column=None,
    
    # UMAP Columns
    umap_x_column='UMAP_1',
    umap_y_column='UMAP_2',
    
    well_column='Metadata_WellID',
    field_column='Metadata_Field',
    
    # Image Settings
    tile_size=300,
    channel_names=['DNA', 'GFP', 'CMO', 'TP63', 'Phalloidin'],
    
    # Filename Pattern
    filename_pattern="{well}_F{field:04d}_T0001_Z0001_C{channel:02d}.tif"
)
# Initialize Widget
from src.umap_exploration_widget import UMAPExplorationWidget
# Ensure df_cell is loaded
umap_widget = UMAPExplorationWidget(df_clean, config)
umap_widget.display()

NameError: name 'df_clean' is not defined

## Automated Tile Extraction by Metadata or Clustering

In [ ]:
# Import the widget
from src.automated_tile_widget import AutomatedTileExportWidget

# If you already have your dataframe loaded in umap_widget, you can pass it directly.
# Otherwise, pass pd.DataFrame() and use the "Load DF" button in the UI.
df_to_use = umap_widget.df if 'umap_widget' in locals() else pd.DataFrame()

# Initialize and display the automated exporter
auto_export_widget = AutomatedTileExportWidget(df=df_to_use, config=config, output_root="UMAP")
auto_export_widget.display()


## See which features drive UMAP separation

In [ ]:
from src.feature_importance_widget import FeatureImportanceWidget
if 'umap_widget' in locals():
    fi_widget = FeatureImportanceWidget(umap_widget.df)
    #fi_widget = FeatureImportanceWidget(umap_widget.df)
    fi_widget.display()
else:
    print("Please run the UMAP Exploration Widget first, or provide a dataframe.")

# Image Based Classifier

In [ ]:
from src.tile_extraction import NotebookConfig
from src.classification_widget import ClassificationWidget
config = NotebookConfig(
    # Data Paths
    csv_path=r"C:\Matthew_Analysis\Airway_Analysis_Classification\TSZ_Screen\CP_CSVs\master_single_cell.csv",
    image_base_path=r"D:\Images\MMC_0023\20260217T092025",
    output_dir='./UMAP',
    
    # Bounding Box Columns (Use these instead of x/y_column)
    bbox_min_x_column='Nucleus_AreaShape_BoundingBoxMinimum_X',
    bbox_max_x_column='Nucleus_AreaShape_BoundingBoxMaximum_X',
    bbox_min_y_column='Nucleus_AreaShape_BoundingBoxMinimum_Y',
    bbox_max_y_column='Nucleus_AreaShape_BoundingBoxMaximum_Y',
    
    # Optional: Center columns (can be None if using bbox)
    x_column=None,
    y_column=None,
    
    # UMAP Columns
    umap_x_column='UMAP_1',
    umap_y_column='UMAP_2',
    
    well_column='Metadata_WellID',
    field_column='Metadata_Field',
    
    # Image Settings
    tile_size=300,
    channel_names=['DNA', 'KRT8', 'CMO', 'TP63', 'Phalloidin'],
    
    # Filename Pattern
    filename_pattern="{well}_F{field:04d}_T0001_Z0001_C{channel:02d}.tif"
)
# Initialize the widget
# 'df' must be your dataframe
# 'config' must be your NotebookConfig object (same as used for UMAP)
clf_widget = ClassificationWidget(df_clean, config, class_names=['TP63_Possitive', 'TP63_Negative'])
clf_widget.display()

## Trial widget for calculating Wasserstein distances between condtions
### Should normalize the data first

In [ ]:
from pycytominer import normalize

normalized_df = normalize(
    profiles=filtered_df,
    method="mad_robustize",
    samples="all" 
)

In [ ]:
df = normalized_df

from src.wasserstein_widget import WassersteinDistanceWidget
# Initialize and display the widget
# We use df_cell as it contains the feature data
if 'df' in locals() and df is not None:
    wasserstein_widget = WassersteinDistanceWidget(df)
    wasserstein_widget.display()
else:
    print("Error: df is not defined. Please load data first.")